# 00 — Démarrer avec le lac de données V2

Ce notebook utilise par défaut la petite fixture déterministe ``data_smoke``. Pour analyser le lac complet, définissez explicitement ``POLY_DATA_ROOT`` et ``POLY_NOTEBOOK_MODE=full``.

In [ ]:
from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context()
store = ParquetStore(ctx.root)
SOURCES = ['order_filled_v2', 'trades', 'markets_current', 'market_assets', 'market_outcomes']
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision})
inventory = source_inventory(store, SOURCES)
inventory

## Le flux de données

``order_filled_v2`` est le bronze : les événements de remplissage V2 avec leur provenance de bloc. ``trades`` est le silver normalisé. Les dimensions ``markets_current`` et ``market_assets`` relient un actif à son marché. Enfin, ``market_outcomes`` conserve les résolutions officielles sans réécrire l'historique.

In [ ]:
from pathlib import Path

for source in SOURCES:
    files = sorted((ctx.root / source).rglob('*.parquet'))
    print(f'{source:18} partitions={len({p.parent for p in files})} files={len(files)}')
print('manifest files:', len(list((ctx.root / '_metadata').rglob('*.json'))))

## Glossaire

- **asset** : jeton d'une issue, relié à un marché par ``market_assets``.
- **token1 / token2** : les deux issues binaires conservées par le contrat.
- **market_outcomes** : étiquette officielle finale ; une absence signifie qu'un marché reste non résolu, pas une issue négative.
- **partition** : dossier ``year=YYYY/month=MM`` qui permet de lire seulement la période utile.